# Study D, Milestone 1: results walkthrough
Decision: **REFORMULATE** (see `reports/milestone1_decision.md`).

In [1]:
import json, sys, numpy as np, pandas as pd
sys.path.insert(0, '..')
from src.bounds.core import tau_min, tau_min_lp, tv, support_mismatch, tau_lower_robust
m = json.load(open('../results/metrics/milestone1_metrics.json'))

## P2: closed form vs independent LP (E1)

In [2]:
pd.Series(m['E1'])

n                             1.000000e+03
max_abs_diff                  1.173395e-12
lam1_equals_tv                4.440892e-16
laminf_equals_sm              1.110223e-16
max_monotonicity_violation    1.701861e-11
tmax_median                   1.000000e+00
tmax_min                      7.710788e-01
closed_d100_s                 3.767014e-04
robustLP_d100_s               8.925438e-03
closed_d1000_s                3.774166e-04
robustLP_d1000_s              5.025768e-01
dtype: float64

## C1: one (P, Q), many mechanisms

In [3]:
P = np.array([.9, .09, .01]); Q = np.array([.4, .3, .3])
pd.DataFrame({'Lambda': L, 'tau_min': tau_min(P, Q, L), 'LP': tau_min_lp(P, Q, L)} for L in [1, 2, 5, 10, 30, 67.5, 100])

,Lambda,tau_min,LP
0,1.0,5.000000e-01,0.500000
1,2.0,4.181818e-01,0.418182
2,5.0,2.631579e-01,0.263158
3,10.0,2.300000e-01,0.230000
4,30.0,1.250000e-01,0.125000
5,67.5,5.218048e-15,0.000000
6,100.0,0.000000e+00,0.000000


## Nontriviality (E2): frozen family and the biologically shaped family

In [4]:
d = pd.read_csv('../results/tables/e2_nontriviality_perturbation_family.csv')
print(m['E2'])
d[d.lam_true <= d.lam].groupby('lam').agg(n=('i', 'size'), frac_positive=('tau_min', lambda x: (x > 1e-9).mean()), tau_true=('tau_true', 'median'))

{'frac_strictly_between_lam2': 0.762, 'frac_strictly_between_lam5': 0.866, 'frac_strictly_between_lam10': 0.825, 'frac_strictly_between_lam100': 0.483}


,n,frac_positive,tau_true
lam,,,
5,49,0.183673,0.132559
10,272,0.040441,0.132388
100,972,0.000000,0.133244


## Scenarios S1-S8 (E4)

In [5]:
e = pd.read_csv('../results/tables/e4_scenarios.csv')
e[['scenario', 'n', 'lam', 'tau_true', 'tau_min_pop', 'plug_pos', 'tv_pos', 'sm_pos', 'robust_pos', 'robust_cover', 'robust_mean']].round(3)

,scenario,n,lam,tau_true,tau_min_pop,plug_pos,tv_pos,sm_pos,robust_pos,robust_cover,robust_mean
0,S1,200,50.000,0.00,0.000,0.768,1.0,0.432,0.000,1.000,0.000
1,S2,200,50.000,0.30,0.300,1.000,1.0,1.000,0.000,1.000,0.000
2,S3,200,10.000,0.12,0.024,0.742,1.0,0.120,0.000,1.000,0.000
3,S6,200,19.896,0.37,0.115,0.948,1.0,0.000,0.015,1.000,0.000
4,S4_p0.0001,200,4285.286,0.00,0.000,0.998,1.0,0.998,0.000,1.000,0.000
5,S4_p0.0005,200,856.714,-0.00,0.000,0.888,1.0,0.888,0.000,1.000,0.000
6,S4_p0.002,200,213.857,-0.00,0.000,0.718,1.0,0.710,0.000,1.000,0.000
7,S1,1000,50.000,0.00,0.000,0.650,1.0,0.008,0.000,1.000,0.000
8,S2,1000,50.000,0.30,0.300,1.000,1.0,1.000,0.965,1.000,0.022
9,S3,1000,10.000,0.12,0.024,0.880,1.0,0.000,0.002,1.000,0.000


In [6]:
pd.read_csv('../results/tables/e4_equivalence_twins.csv')

,scenario,tau_true,tau_twin,maxdiff_PQ,robust_identical,robust_mean,cover_twin
0,S1,2.220446e-16,0.788268,5.551115e-17,True,0.000000,1.0
1,S3,1.200000e-01,0.738462,1.110223e-16,True,0.000024,1.0
2,S4_p5e-4,-2.220446e-16,0.814443,5.551115e-17,True,0.000000,1.0


In [7]:
pd.Series(m['E4_S5'])

tau_true                    1.897282e-01
clone_tau_min_population    1.744646e-01
eps0.0_clone_cover          1.000000e+00
eps0.0_clone_mean           8.181008e-02
eps0.0_pop5_mean            7.654821e-02
eps0.0_pop5000_mean         0.000000e+00
eps0.05_clone_cover         1.000000e+00
eps0.05_clone_mean          3.329350e-02
eps0.05_pop5_mean           7.619303e-02
eps0.05_pop5000_mean        0.000000e+00
eps0.2_clone_cover          1.000000e+00
eps0.2_clone_mean           1.418256e-02
eps0.2_pop5_mean            7.631256e-02
eps0.2_pop5000_mean         0.000000e+00
null_clone_pos              0.000000e+00
null_pop_lam5_pos           1.000000e+00
s5_null_tau                -3.469447e-18
dtype: float64

## Design (E5) and founder bottleneck (E6)

In [8]:
pd.read_csv('../results/tables/e5_design_required_n.csv').pivot_table(index=['q', 'rho'], columns='lam', values='n_required_cells')

lam        2        10        100        1000        10000
q    rho                                                  
0.05 1     238.0   1177.0   11734.0   117306.0   1173023.0
     10   2350.0  11734.0  117306.0  1173023.0  11730197.0
0.10 1     115.0    559.0    5560.0    55568.0    555644.0
     10   1115.0   5560.0   55568.0   555644.0   5556411.0
0.30 1      32.0    148.0    1444.0    14409.0    144059.0
     10    292.0   1444.0   14409.0   144059.0   1440554.0

In [9]:
pd.read_csv('../results/tables/e5_regional_bias.csv')

,rho,n,false_induction
0,1,2000,0.00
1,1,20000,0.00
2,3,2000,0.06
3,3,20000,1.00
4,10,2000,0.61
5,10,20000,1.00


In [10]:
pd.read_csv('../results/tables/e6_single_met_neutral.csv').pivot(index='nf', columns='lam', values='false_induction')

lam,1.0,10.0,100.0
nf,,,
1,1.00,1.00,1.00
3,1.00,1.00,0.78
10,1.00,0.89,0.14
100,0.95,0.00,0.00
1000,0.21,0.00,0.00


In [11]:
pd.read_csv('../results/tables/e6_fluctuation_tests.csv').round(4)

,nf,ks_p_sel_vs_founderind,ks_p_sel_vs_cellind,var_sel,var_founder,var_cell
0,1,0.9997,0.0,0.2143,0.1696,0.0001
1,3,0.5830,0.0,0.0779,0.0755,0.0001
2,10,0.9942,0.0,0.0265,0.0279,0.0001
3,100,0.8154,0.0,0.0031,0.0031,0.0001


## Dynamics (E7)

In [12]:
pd.read_csv('../results/tables/e7_dynamics_fits.csv').groupby(['design', 'ncell'])[['a', 'c', 'db']].median().round(3)

a      c     db
design      ncell                     
2 snapshots 0      0.049  0.009  0.958
            2000   0.152  0.100  0.104
            20000  0.143  0.100  0.104
4 snapshots 0      0.080  0.020  0.600
            2000   0.079  0.024  0.586
            20000  0.080  0.017  0.589
8 snapshots 0      0.080  0.020  0.600
            2000   0.082  0.014  0.581
            20000  0.080  0.020  0.602